# Fig. 6 复现偏差：分层诊断

本 Notebook 不修改主体物理模块，而是同时进行两条检查：

1. **图像轨**：直接读取论文 PDF 第 5 页的矢量路径，恢复 Fig. 6 的两条捕获率曲线和红色质量轨迹；
2. **公式轨**：把当前每晕捕获率拆成晕密度积分、速度平均和 PBH 质量核，逐层比较。

论文中 `z=0` 数据点位于对数横轴之外，所以 PDF 中每条路径虽然有 50 个顶点，第一个顶点实际上是被坐标轴裁剪后的边界交点。Notebook 会保留这一行，但把无法可靠反算的源图数值记为 `NaN`；可可靠恢复的是其余 49 个物理数据点。

In [ ]:
from pathlib import Path
import csv
import sys

import matplotlib.pyplot as plt
import numpy as np
import pymupdf
from scipy.integrate import quad

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'twobodycapture.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import haloconcentration as hc
import halostructure as hs
import pbhmassfunction as pmf
import twobodycapture as capture

PAPER_PATH = PROJECT_ROOT / 'Simulating Binary Primordial Black Hole Mergers in Dark Matter Halos.pdf'
REDSHIFT = np.linspace(0.0, 12.0, 50)
POSITIVE_Z = REDSHIFT > 0.0
PRESENT_DAY_MASS_MSUN = 1.0e12
MONOCHROMATIC_MASS_MSUN = 30.0

print('项目目录：', PROJECT_ROOT)
print('论文 PDF：', PAPER_PATH)

## 1. 从 PDF 矢量路径恢复源图

这里不是对 PNG 像素取样，而是读取 Matplotlib 写入 PDF 的折线路径。颜色、路径点数和所在面板共同确定目标曲线。坐标转换使用图中的主刻度和次刻度，因此不会用当前程序的物理结果去校准源图。

In [ ]:
TAB_BLUE = np.array([0.1215686277, 0.4666666687, 0.7058823705])
PURE_BLUE = np.array([0.0, 0.0, 1.0])
PURE_ORANGE = np.array([1.0, 0.6470588446, 0.0])
PURE_GREEN = np.array([0.0, 0.5019607544, 0.0])
TAB_GREEN = np.array([0.1725490242, 0.6274510026, 0.1725490242])
TAB_RED = np.array([0.8392156959, 0.1529411823, 0.1568627506])

def drawing_vertices(drawing):
    line_items = [item for item in drawing['items'] if item[0] == 'l']
    points = [line_items[0][1]] + [item[2] for item in line_items]
    return np.array([[point.x, point.y] for point in points], dtype=float)

def find_colored_curve(drawings, color, segment_count, x_range, y_range):
    candidates = []
    for drawing in drawings:
        drawing_color = drawing.get('color')
        rectangle = drawing['rect']
        if drawing_color is None:
            continue
        if not np.allclose(drawing_color, color, atol=2.0e-5):
            continue
        if len(drawing['items']) != segment_count:
            continue
        if not (x_range[0] <= rectangle.x0 <= x_range[1]):
            continue
        if not (y_range[0] <= rectangle.y0 <= y_range[1]):
            continue
        candidates.append(drawing)
    if len(candidates) != 1:
        raise RuntimeError(f'目标矢量曲线候选数为 {len(candidates)}，无法唯一识别。')
    return candidates[0]

paper = pymupdf.open(PAPER_PATH)
page5_drawings = paper[4].get_drawings()

fig6_lognormal_path = find_colored_curve(
    page5_drawings, PURE_ORANGE, 49, (300.0, 340.0), (440.0, 480.0)
)
fig6_mono_path = find_colored_curve(
    page5_drawings, PURE_BLUE, 49, (300.0, 340.0), (450.0, 490.0)
)
fig6_mass_path = find_colored_curve(
    page5_drawings, TAB_RED, 49, (300.0, 340.0), (470.0, 500.0)
)

fig6_lognormal_xy = drawing_vertices(fig6_lognormal_path)
fig6_mono_xy = drawing_vertices(fig6_mono_path)
fig6_mass_xy = drawing_vertices(fig6_mass_path)

# 横轴主刻度 z=1 和 z=10 的矢量坐标。
X_AT_Z1 = 419.661438
X_AT_Z10 = 516.100464
source_z_from_x = 10.0 ** (
    (fig6_mono_xy[:, 0] - X_AT_Z1) / (X_AT_Z10 - X_AT_Z1)
)
source_z_from_x[0] = np.nan

# 左纵轴：10^-12 主刻度，以及 2*10^-12 次刻度。
Y_RATE_1E_MINUS12 = 536.528809
Y_RATE_2E_MINUS12 = 508.323364
RATE_POINTS_PER_DECADE = (
    (Y_RATE_1E_MINUS12 - Y_RATE_2E_MINUS12) / np.log10(2.0)
)

def rate_from_pdf_y(y_coordinate):
    return 10.0 ** (
        -12.0
        + (Y_RATE_1E_MINUS12 - y_coordinate) / RATE_POINTS_PER_DECADE
    )

source_mono_rate = rate_from_pdf_y(fig6_mono_xy[:, 1])
source_lognormal_rate = rate_from_pdf_y(fig6_lognormal_xy[:, 1])

# 右纵轴相邻十倍质量主刻度的间距。
Y_MASS_1E12 = 480.346008
MASS_POINTS_PER_DECADE = 28.433502
source_mass_msun = 10.0 ** (
    12.0 + (Y_MASS_1E12 - fig6_mass_xy[:, 1]) / MASS_POINTS_PER_DECADE
)

# 第一个顶点是坐标轴裁剪交点，不是可恢复的 z=0 数据点。
source_mono_rate[0] = np.nan
source_lognormal_rate[0] = np.nan
source_mass_msun[0] = np.nan

z_coordinate_difference = source_z_from_x[1:] - REDSHIFT[1:]
print('Fig. 6 路径顶点数：', len(fig6_mono_xy))
print('49 个可恢复点的最大红移坐标误差：', np.max(np.abs(z_coordinate_difference)))

In [ ]:
# Fig. 1 的蓝色虚线给出同一 M0=10^12 M_sun 晕的源图浓度轨迹。
page3_drawings = paper[2].get_drawings()
fig1_concentration_path = find_colored_curve(
    page3_drawings, TAB_BLUE, 99, (40.0, 70.0), (60.0, 90.0)
)
fig1_concentration_xy = drawing_vertices(fig1_concentration_path)
fig1_z = np.linspace(0.0, 12.0, 100)

# Fig. 1 左轴是线性轴；C=8 与 C=7 的刻度位置如下。
Y_AT_C8 = 68.240417
CONCENTRATION_POINTS_PER_UNIT = 23.945465
fig1_source_concentration = (
    8.0
    + (Y_AT_C8 - fig1_concentration_xy[:, 1])
    / CONCENTRATION_POINTS_PER_UNIT
)
fig1_source_concentration[0] = np.nan
source_concentration_at_fig6_z = np.full_like(REDSHIFT, np.nan)
source_concentration_at_fig6_z[1:] = np.interp(
    REDSHIFT[1:], fig1_z[1:], fig1_source_concentration[1:]
)

# Fig. 3、Fig. 4 的纵轴每十倍间距相同。这里只需曲线间距，不需绝对纵轴零点。
RATE_POINTS_PER_DECADE_FIG3_FIG4 = 29.615661
fig3_mono = drawing_vertices(
    find_colored_curve(page5_drawings, TAB_BLUE, 49, (80.0, 110.0), (60.0, 90.0))
)
fig3_lognormal = drawing_vertices(
    find_colored_curve(page5_drawings, TAB_GREEN, 49, (80.0, 110.0), (60.0, 90.0))
)
fig4_mono = drawing_vertices(
    find_colored_curve(page5_drawings, PURE_BLUE, 49, (80.0, 110.0), (380.0, 410.0))
)
fig4_lognormal_sigma06 = drawing_vertices(
    find_colored_curve(page5_drawings, PURE_GREEN, 49, (80.0, 110.0), (380.0, 410.0))
)

def median_vertical_ratio(higher_curve_xy, lower_curve_xy, points_per_decade):
    return np.median(10.0 ** (
        (lower_curve_xy[:, 1] - higher_curve_xy[:, 1])
        / points_per_decade
    ))

source_ratio_fig3 = median_vertical_ratio(
    fig3_lognormal, fig3_mono, RATE_POINTS_PER_DECADE_FIG3_FIG4
)
source_ratio_fig4 = median_vertical_ratio(
    fig4_lognormal_sigma06, fig4_mono, RATE_POINTS_PER_DECADE_FIG3_FIG4
)
source_ratio_fig6 = np.nanmedian(source_lognormal_rate / source_mono_rate)

source_vector_table = np.column_stack([
    REDSHIFT,
    source_z_from_x,
    source_mono_rate,
    source_lognormal_rate,
    source_mass_msun,
    source_concentration_at_fig6_z,
])
source_vector_path = PROJECT_ROOT / 'fig6_source_vector.csv'
np.savetxt(
    source_vector_path,
    source_vector_table,
    delimiter=',',
    header=(
        'z_grid,z_recovered_from_pdf_x,source_mono_rate_per_year,'
        'source_lognormal_rate_per_year,source_mass_msun,'
        'source_concentration_from_fig1'
    ),
    comments='',
    fmt='%.12e',
)
print('源图数据：', source_vector_path)
print('Fig. 3/4/6 矢量曲线的 log-normal/mono 倍率：',
      source_ratio_fig3, source_ratio_fig4, source_ratio_fig6)

## 2. 当前代码基线与物理量分解

单色谱下当前代码满足

$$R_{\rm halo}=\frac12 I_{\rho^2}\,\frac{\langle\Sigma v\rangle}{m^2},$$

其中 $I_{\rho^2}=4\pi\int r^2\rho_{\rm NFW}^2dr$。下面把每一项单独保存，便于观察是哪一项控制峰值。

In [ ]:
monochromatic_nodes, monochromatic_weights = (
    pmf.monochromatic_mass_distribution(MONOCHROMATIC_MASS_MSUN)
)

def lognormal_sigma06_pdf(mass_msun):
    return pmf.lognormal_mass_pdf(
        mass_msun, median_mass_msun=30.0, sigma=0.6
    )

lognormal_nodes, lognormal_weights = pmf.continuous_mass_quadrature(
    lognormal_sigma06_pdf, point_count=96
)

current_mass_msun = hc.mass_accretion_history(
    PRESENT_DAY_MASS_MSUN, REDSHIFT, 'ludlow16'
)
current_concentration = hc.concentration_model(
    current_mass_msun, REDSHIFT, 'ludlow16'
)

current_mono_rate = np.array([
    capture.capture_rate_per_halo_a14_per_year(
        mass, z, monochromatic_nodes, monochromatic_weights,
        concentration_model='ludlow16', f_pbh=1.0,
    )
    for mass, z in zip(current_mass_msun, REDSHIFT)
])
current_lognormal_rate = np.array([
    capture.capture_rate_per_halo_a14_per_year(
        mass, z, lognormal_nodes, lognormal_weights,
        concentration_model='ludlow16', f_pbh=1.0,
    )
    for mass, z in zip(current_mass_msun, REDSHIFT)
])

def halo_components(mass_msun, z, concentration):
    rho_critical = float(hs.rho_crit(z, hc.LUDLOW_COSMOLOGY))
    virial_radius = (
        3.0 * mass_msun / (4.0 * np.pi * 200.0 * rho_critical)
    ) ** (1.0 / 3.0)
    scale_radius = virial_radius / concentration
    scale_density = float(
        hs.rho_s(concentration, z, cosmology=hc.LUDLOW_COSMOLOGY)
    )
    velocity_dispersion = float(
        hs.nfw_characteristic_velocity(scale_radius, scale_density)
    )
    cutoff_velocity = float(
        hs.virial_cutoff_velocity(mass_msun, virial_radius)
    )
    concentration_factor = 1.0 - 1.0 / (1.0 + concentration) ** 3
    density_squared_integral = (
        mass_msun**2 * concentration_factor
        / (
            12.0 * np.pi * scale_radius**3
            * hc.nfw_g(concentration) ** 2
        )
    )
    sigma_v = float(
        capture.velocity_averaged_capture_coefficient_kpc3_per_year(
            MONOCHROMATIC_MASS_MSUN,
            MONOCHROMATIC_MASS_MSUN,
            velocity_dispersion,
            cutoff_velocity,
        )
    )
    reconstructed_rate = (
        0.5 * density_squared_integral * sigma_v
        / MONOCHROMATIC_MASS_MSUN**2
    )
    return (
        rho_critical, virial_radius, scale_radius, scale_density,
        velocity_dispersion, cutoff_velocity, density_squared_integral,
        sigma_v, reconstructed_rate,
    )

component_rows = np.array([
    halo_components(mass, z, concentration)
    for mass, z, concentration in zip(
        current_mass_msun, REDSHIFT, current_concentration
    )
])
(
    rho_critical, virial_radius, scale_radius, scale_density,
    velocity_dispersion, cutoff_velocity, density_squared_integral,
    sigma_v, reconstructed_rate,
) = component_rows.T

dlog_density_dz = np.gradient(np.log(density_squared_integral), REDSHIFT)
dlog_sigma_v_dz = np.gradient(np.log(sigma_v), REDSHIFT)
dlog_rate_dz = dlog_density_dz + dlog_sigma_v_dz

component_table = np.column_stack([
    REDSHIFT, current_mass_msun, current_concentration, rho_critical,
    virial_radius, scale_radius, scale_density, velocity_dispersion,
    cutoff_velocity, density_squared_integral, sigma_v,
    reconstructed_rate, current_mono_rate,
    dlog_density_dz, dlog_sigma_v_dz, dlog_rate_dz,
])
component_path = PROJECT_ROOT / 'fig6_diagnostic_components.csv'
np.savetxt(
    component_path, component_table, delimiter=',', comments='', fmt='%.12e',
    header=(
        'z,mass_msun,concentration,rho_crit_msun_kpc3,Rvir_kpc,Rs_kpc,'
        'rho_s_msun_kpc3,vdisp_km_s,vvir_km_s,I_rho2_msun2_kpc3,'
        'sigma_v_kpc3_per_year,reconstructed_rate_per_year,'
        'module_rate_per_year,dln_Irho2_dz,dln_sigma_v_dz,dln_rate_dz'
    ),
)
print('物理量分解：', component_path)
print('分解率 / 模块率的范围：',
      np.min(reconstructed_rate / current_mono_rate),
      np.max(reconstructed_rate / current_mono_rate))

## 3. 单变量控制实验：质量史、浓度和质量单位

这里所有替换都只发生在 Notebook 内：

- 使用 Fig. 9 矢量轨迹反求的 $(\alpha,\beta)$；
- 只替换 Fig. 6 的红色质量轨迹；
- 只替换 Fig. 1 的蓝色浓度轨迹；
- 同时替换源图质量和浓度；
- 检查把物理质量直接当作 $h^{-1}M_\odot$ 数值时的浓度分支。

最后一个分支只用于定位单位约定，不代表接受这种写法。

In [ ]:
def rate_track_from_explicit_state(mass_track, concentration_track):
    return np.array([
        halo_components(mass, z, concentration)[-1]
        for mass, z, concentration in zip(
            mass_track, REDSHIFT, concentration_track
        )
    ])

def peak_metrics(z, rate):
    valid = np.isfinite(rate) & (rate > 0.0)
    zv = z[valid]
    rv = rate[valid]
    peak_index = int(np.argmax(rv))
    discrete_peak_z = float(zv[peak_index])
    maximum_rate = float(rv[peak_index])

    lower = max(0, peak_index - 2)
    upper = min(len(zv), peak_index + 3)
    coefficients = np.polyfit(zv[lower:upper], np.log(rv[lower:upper]), 2)
    quadratic_peak_z = discrete_peak_z
    if coefficients[0] < 0.0:
        candidate = -coefficients[1] / (2.0 * coefficients[0])
        if zv[lower] <= candidate <= zv[upper - 1]:
            quadratic_peak_z = float(candidate)

    plateau = zv[rv >= 0.99 * maximum_rate]
    rate_at_z5 = float(np.interp(5.0, zv, rv))
    return {
        'peak_discrete_z': discrete_peak_z,
        'peak_quadratic_z': quadratic_peak_z,
        'plateau_z_min': float(plateau[0]),
        'plateau_z_max': float(plateau[-1]),
        'rate_z5_over_peak': rate_at_z5 / maximum_rate,
        'maximum_rate_per_year': maximum_rate,
    }

source_mass_filled = source_mass_msun.copy()
source_mass_filled[0] = PRESENT_DAY_MASS_MSUN
source_concentration_filled = source_concentration_at_fig6_z.copy()
source_concentration_filled[0] = current_concentration[0]

fig9_alpha = 0.233840
fig9_beta = -0.770741
fig9_mass_msun = (
    PRESENT_DAY_MASS_MSUN
    * (1.0 + REDSHIFT) ** fig9_alpha
    * np.exp(fig9_beta * REDSHIFT)
)
fig9_concentration = hc.concentration_model(
    fig9_mass_msun, REDSHIFT, 'ludlow16'
)
source_mass_current_c = hc.concentration_model(
    source_mass_filled, REDSHIFT, 'ludlow16'
)

# 若公式期待输入的数值本身就是 h^-1 M_sun，公开的物理质量会等效地除以 h。
no_h_conversion_concentration = hc.concentration_model(
    current_mass_msun / hc.LUDLOW_COSMOLOGY['h'],
    REDSHIFT,
    'ludlow16',
)

rate_branches = {
    'source vector target': source_mono_rate,
    'current baseline': current_mono_rate,
    'Fig9 alpha-beta + current C': rate_track_from_explicit_state(
        fig9_mass_msun, fig9_concentration
    ),
    'Fig6 source M + current C': rate_track_from_explicit_state(
        source_mass_filled, source_mass_current_c
    ),
    'current M + Fig1 source C': rate_track_from_explicit_state(
        current_mass_msun, source_concentration_filled
    ),
    'Fig6 source M + Fig1 source C': rate_track_from_explicit_state(
        source_mass_filled, source_concentration_filled
    ),
    'current M + no-h-conversion C': rate_track_from_explicit_state(
        current_mass_msun, no_h_conversion_concentration
    ),
}

branch_metrics = {name: peak_metrics(REDSHIFT, rate)
                  for name, rate in rate_branches.items()}
for name, metrics in branch_metrics.items():
    print(name, metrics)

In [ ]:
figure, axes = plt.subplots(3, 1, figsize=(8.2, 11.0), sharex=True)

axes[0].plot(REDSHIFT[POSITIVE_Z], source_mono_rate[POSITIVE_Z],
             'o-', color='black', markersize=3.2, linewidth=1.4,
             label='PDF vector: mono.')
axes[0].plot(REDSHIFT[POSITIVE_Z], current_mono_rate[POSITIVE_Z],
             '--', color='tab:blue', linewidth=2.0, label='current baseline')
axes[0].plot(REDSHIFT[POSITIVE_Z],
             rate_branches['Fig6 source M + Fig1 source C'][POSITIVE_Z],
             '-.', color='tab:green', linewidth=1.8,
             label='source M + source C diagnostic')
axes[0].set_yscale('log')
axes[0].set_ylabel(r'$R_{\rm halo}\;[\mathrm{yr}^{-1}]$')
axes[0].legend(fontsize=8)
axes[0].grid(True, which='both', alpha=0.25)

axes[1].plot(REDSHIFT[POSITIVE_Z], source_mass_msun[POSITIVE_Z],
             'o-', color='black', markersize=3.0, label='PDF Fig. 6 mass')
axes[1].plot(REDSHIFT[POSITIVE_Z], current_mass_msun[POSITIVE_Z],
             '--', color='tab:red', linewidth=2.0, label='current Appendix C')
axes[1].plot(REDSHIFT[POSITIVE_Z], fig9_mass_msun[POSITIVE_Z],
             ':', color='tab:purple', linewidth=2.0,
             label='recovered Fig. 9 alpha-beta')
axes[1].set_yscale('log')
axes[1].set_ylabel(r'$M(z)\;[M_\odot]$')
axes[1].legend(fontsize=8)
axes[1].grid(True, which='both', alpha=0.25)

axes[2].plot(REDSHIFT[POSITIVE_Z],
             source_concentration_at_fig6_z[POSITIVE_Z],
             'o-', color='black', markersize=3.0, label='PDF Fig. 1 C(z)')
axes[2].plot(REDSHIFT[POSITIVE_Z], current_concentration[POSITIVE_Z],
             '--', color='tab:blue', linewidth=2.0, label='current C[M(z),z]')
axes[2].plot(REDSHIFT[POSITIVE_Z],
             no_h_conversion_concentration[POSITIVE_Z],
             ':', color='tab:orange', linewidth=2.0,
             label='diagnostic: no h conversion')
axes[2].set_xscale('log')
axes[2].set_xlabel('Redshift z')
axes[2].set_ylabel('Concentration C')
axes[2].legend(fontsize=8)
axes[2].grid(True, which='both', alpha=0.25)

figure.suptitle('Fig. 6 layered diagnosis: source vectors and current model')
figure.tight_layout()
overlay_path = PROJECT_ROOT / 'fig6_diagnostic_overlay.png'
figure.savefig(overlay_path, dpi=240, bbox_inches='tight')
print('叠加图：', overlay_path)
plt.show()

In [ ]:
valid_source = POSITIVE_Z & np.isfinite(source_mono_rate)
source_mc_rate = rate_branches['Fig6 source M + Fig1 source C']

figure, axes = plt.subplots(2, 1, figsize=(8.2, 7.8), sharex=True)
axes[0].axhline(0.0, color='0.4', linewidth=1.0)
axes[0].plot(
    REDSHIFT[valid_source],
    np.log10(current_mono_rate[valid_source] / source_mono_rate[valid_source]),
    color='tab:blue', linewidth=2.0, label='current - PDF vector',
)
axes[0].plot(
    REDSHIFT[valid_source],
    np.log10(source_mc_rate[valid_source] / source_mono_rate[valid_source]),
    color='tab:green', linestyle='-.', linewidth=1.8,
    label='source M+C diagnostic - PDF vector',
)
axes[0].set_ylabel(r'$\Delta\log_{10}R$')
axes[0].legend(fontsize=8)
axes[0].grid(True, which='both', alpha=0.25)

axes[1].axhline(0.0, color='0.4', linewidth=1.0)
axes[1].plot(REDSHIFT[POSITIVE_Z], dlog_density_dz[POSITIVE_Z],
             label=r'$d\ln I_{\rho^2}/dz$', linewidth=2.0)
axes[1].plot(REDSHIFT[POSITIVE_Z], dlog_sigma_v_dz[POSITIVE_Z],
             label=r'$d\ln\langle\Sigma v\rangle/dz$', linewidth=2.0)
axes[1].plot(REDSHIFT[POSITIVE_Z], dlog_rate_dz[POSITIVE_Z],
             label='sum = d ln R / dz', color='black', linewidth=1.7)
axes[1].set_xscale('log')
axes[1].set_xlabel('Redshift z')
axes[1].set_ylabel('Logarithmic derivative')
axes[1].legend(fontsize=8)
axes[1].grid(True, which='both', alpha=0.25)

figure.tight_layout()
residual_path = PROJECT_ROOT / 'fig6_diagnostic_residuals.png'
figure.savefig(residual_path, dpi=240, bbox_inches='tight')
print('残差与导数图：', residual_path)
plt.show()

## 4. 速度分布的三种阅读方式

当前实现把 $p(v)$ 当作三维各向同性速度密度，使用 $4\pi v^2dv$ 测度。这与 Appendix A8-A12 的三维积分结构及捕获截面的 $v^{-11/7}$ 标度相容。

论文 Eq. (7) 的印刷式在 $p(v)$ 中又显式出现 $v^2$；如果把它解释成已经归一化的速率 PDF，再直接放入 Eq. (17)，速度矩会从 $v^{-11/7}$ 型变成 $v^{+3/7}$ 型，而且不能直接保持 A14 的量纲。因此下面只比较这种字面解释的**红移形状**，并在 $z=0$ 人为归一到基线，绝不把它当作新的绝对捕获率。Eq. (17) 中的额外因子 2 只会给当前分支乘上常数 $2^{3/7}$，不会移动峰值。

In [ ]:
def velocity_moments(velocity_dispersion, cutoff_velocity):
    def f3(speed):
        return float(hs.truncated_maxwell_velocity_density(
            speed, velocity_dispersion, cutoff_velocity
        ))

    appendix_moment, _ = quad(
        lambda speed: 4.0 * np.pi * speed ** (3.0 / 7.0) * f3(speed),
        0.0, cutoff_velocity, epsabs=0.0, epsrel=1.0e-9, limit=200,
    )
    literal_speed_pdf_moment, _ = quad(
        lambda speed: (
            4.0 * np.pi * speed**2 * f3(speed)
            * speed ** (3.0 / 7.0)
        ),
        0.0, cutoff_velocity, epsabs=0.0, epsrel=1.0e-9, limit=200,
    )
    return appendix_moment, literal_speed_pdf_moment

velocity_moment_rows = np.array([
    velocity_moments(vdisp, vcut)
    for vdisp, vcut in zip(velocity_dispersion, cutoff_velocity)
])
appendix_velocity_moment = velocity_moment_rows[:, 0]
literal_eq7_moment = velocity_moment_rows[:, 1]

literal_eq7_shape_rate = current_mono_rate[0] * (
    density_squared_integral * literal_eq7_moment
) / (density_squared_integral[0] * literal_eq7_moment[0])
eq17_factor2_rate = current_mono_rate * 2.0 ** (3.0 / 7.0)

velocity_branch_metrics = {
    'Appendix A8-A12 / current 3D': peak_metrics(REDSHIFT, current_mono_rate),
    'Eq7 literal speed-PDF shape': peak_metrics(REDSHIFT, literal_eq7_shape_rate),
    'Eq17 extra factor 2': peak_metrics(REDSHIFT, eq17_factor2_rate),
}
for name, metrics in velocity_branch_metrics.items():
    print(name, metrics)

reference_index = 1
figure, axis = plt.subplots(figsize=(8.0, 5.3))
axis.plot(REDSHIFT[POSITIVE_Z],
          source_mono_rate[POSITIVE_Z] / source_mono_rate[reference_index],
          'o-', color='black', markersize=3, label='PDF vector shape')
axis.plot(REDSHIFT[POSITIVE_Z],
          current_mono_rate[POSITIVE_Z] / current_mono_rate[reference_index],
          '--', linewidth=2, label='Appendix 3D / current')
axis.plot(REDSHIFT[POSITIVE_Z],
          literal_eq7_shape_rate[POSITIVE_Z]
          / literal_eq7_shape_rate[reference_index],
          '-.', linewidth=2, label='literal Eq. 7 speed-PDF shape')
axis.set_xscale('log')
axis.set_yscale('log')
axis.set_xlabel('Redshift z')
axis.set_ylabel(r'$R(z)/R(z_1)$')
axis.legend(fontsize=8)
axis.grid(True, which='both', alpha=0.25)
figure.tight_layout()
velocity_path = PROJECT_ROOT / 'fig6_diagnostic_velocity_branches.png'
figure.savefig(velocity_path, dpi=240, bbox_inches='tight')
print('速度分支图：', velocity_path)
plt.show()

## 5. log-normal/mono 倍率：与红移无关的质量核

Appendix A14 的质量部分是

$$K(m_1,m_2)=\frac{(m_1+m_2)^{10/7}}{(m_1m_2)^{5/7}}.$$

晕结构和速度矩对两种质量函数完全相同，因此质量函数倍率应是常数。下面同时比较当前积分、未进行有限区间重归一化的印刷 log-normal，以及 Fig. 3、4、6 的 PDF 矢量间距。

In [ ]:
def mass_kernel_average(nodes, weights):
    mass1 = nodes[:, np.newaxis]
    mass2 = nodes[np.newaxis, :]
    kernel = (mass1 + mass2) ** (10.0 / 7.0) / (
        mass1 * mass2
    ) ** (5.0 / 7.0)
    return float(np.sum(weights[:, np.newaxis] * weights[np.newaxis, :] * kernel))

mono_kernel = 2.0 ** (10.0 / 7.0)
normalized_lognormal_kernel = mass_kernel_average(
    lognormal_nodes, lognormal_weights
)
normalized_kernel_ratio = normalized_lognormal_kernel / mono_kernel

def raw_printed_lognormal(mass_msun):
    mu = np.log(30.0)
    sigma = 0.6
    return np.exp(-(np.log(mass_msun) - mu) ** 2 / (2.0 * sigma**2)) / (
        np.sqrt(2.0 * np.pi) * sigma * mass_msun
    )

raw_nodes, raw_weights = pmf.continuous_mass_quadrature(
    raw_printed_lognormal, point_count=96
)
raw_kernel_ratio = mass_kernel_average(raw_nodes, raw_weights) / mono_kernel
current_rate_ratio = np.median(current_lognormal_rate / current_mono_rate)

ratio_rows = [
    ('current normalized kernel, 5-150 Msun', normalized_kernel_ratio),
    ('current full-rate ratio', current_rate_ratio),
    ('raw printed PDF without interval renormalization', raw_kernel_ratio),
    ('PDF vector Fig. 3', source_ratio_fig3),
    ('PDF vector Fig. 4, sigma=0.6', source_ratio_fig4),
    ('PDF vector Fig. 6', source_ratio_fig6),
    ('paper prose claim', 3.0),
]
ratio_table_path = PROJECT_ROOT / 'fig6_mass_ratio_diagnostic.csv'
with ratio_table_path.open('w', newline='', encoding='utf-8') as stream:
    writer = csv.writer(stream)
    writer.writerow(['case', 'lognormal_over_monochromatic'])
    writer.writerows(ratio_rows)

print('有限区间内 raw log-normal 的总权重：', np.sum(raw_weights))
for row in ratio_rows:
    print(row)
print('倍率表：', ratio_table_path)

## 6. 因果表与诊断结论

因果表把每个候选解释对峰值、归一化和质量函数倍率的作用放到一起。`source M`、`source C` 等分支只是控制实验，不是建议写入主体模块的经验修正。

In [ ]:
source_metrics = branch_metrics['source vector target']
causal_rows = []

def add_causal_row(name, rate, dimensional, mass_ratio, cross_figure_note):
    metrics = peak_metrics(REDSHIFT, rate)
    comparison_mask = valid_source & np.isfinite(rate)
    median_normalization_ratio = float(np.median(
        rate[comparison_mask] / source_mono_rate[comparison_mask]
    ))
    causal_rows.append([
        name,
        metrics['peak_discrete_z'],
        metrics['peak_quadratic_z'],
        metrics['peak_discrete_z'] - source_metrics['peak_discrete_z'],
        metrics['plateau_z_min'],
        metrics['plateau_z_max'],
        metrics['rate_z5_over_peak'],
        median_normalization_ratio,
        mass_ratio,
        dimensional,
        cross_figure_note,
    ])

for name, rate in rate_branches.items():
    if name == 'source vector target':
        continue
    add_causal_row(
        name, rate, 'yes', normalized_kernel_ratio,
        'mass-function ratio remains consistent with Figs. 3, 4, 6',
    )

add_causal_row(
    'Eq7 literal speed-PDF shape', literal_eq7_shape_rate, 'no',
    normalized_kernel_ratio,
    'shape-only branch; cannot replace A14 without repairing dimensions',
)
add_causal_row(
    'Eq17 extra factor 2', eq17_factor2_rate, 'yes',
    normalized_kernel_ratio,
    'constant 2^(3/7): normalization changes, peak does not',
)

causal_table_path = PROJECT_ROOT / 'fig6_diagnostic_causal_table.csv'
with causal_table_path.open('w', newline='', encoding='utf-8') as stream:
    writer = csv.writer(stream)
    writer.writerow([
        'candidate', 'peak_discrete_z', 'peak_quadratic_z',
        'delta_peak_z_from_source', 'plateau_z_min', 'plateau_z_max',
        'rate_z5_over_peak', 'median_rate_over_source',
        'lognormal_over_mono', 'dimensionally_consistent',
        'fig3_fig4_fig6_note',
    ])
    writer.writerows(causal_rows)

print('源图峰值指标：', source_metrics)
print('因果表：', causal_table_path)
for row in causal_rows:
    print(row)

### 本次运行得到的诊断结论

1. **论文正文的两个数字都不能直接当作原图数据。** PDF 矢量曲线的离散峰值为 $z=3.43$，局部二次拟合为 $z=3.44$；并不是正文所称的 $z\simeq5$。源图的 99% 平台为 $2.94\le z\le4.16$，而 $R(z=5)/R_{\max}=0.957$，所以正文的 5 可以理解成对宽峰的粗略描述，但不是曲线最大点。
2. **“log-normal 比单色谱高 3 倍”是明确的图文冲突。** 当前质量核给出 1.1351746，Fig. 3、Fig. 4 和 Fig. 6 的矢量间距分别给出 1.1351750、1.1351750 和 1.1351747。三幅图与 A14 质量核彼此一致，不支持额外乘 3。
3. **当前峰值偏早主要来自浓度轨迹，不来自质量吸积史。** 当前离散峰值是 $z=2.45$；只换成 Fig. 6 红色质量轨迹仍是 $z=2.45$。只换成 Fig. 1 的源图浓度后峰值变为 $z=3.92$；同时替换源图质量和浓度得到 $z=3.67$，已经接近源图 $z=3.43$。
4. **Fig. 9 反求质量参数不能修复图6。** 代入 $(\alpha,\beta)=(0.233840,-0.770741)$ 后峰值反而提前至 $z=2.20$。因此 Fig. 9 已知的质量史差异不是本问题的主要原因。
5. **质量单位约定值得下一步单独追查，但不能单独解释全部差异。** 暂时跳过 $h$ 转换会把峰值从 $z=2.45$ 推到 $z=2.69$，并让浓度更接近 Fig. 1，但仍不足以恢复源图。这个结果只能说明单位入口是候选原因，不能据此改写公开接口。
6. **当前速度分支应保留。** Appendix A8-A12 的三维速度密度分支量纲闭合；把 Eq. (7) 字面解释成速率 PDF 会把峰值提前到 $z=1.47$，明显恶化形状且无法直接保持 A14 的量纲。Eq. (17) 的额外因子 2 只把率乘以 $2^{3/7}$，不改变峰值。
7. **最小修复建议不是修改 A14。** 下一步应专门核对 Fig. 1 的 Ludlow16 浓度生成口径，包括质量的 $h^{-1}M_\odot$ 输入约定、峰高定义以及作者是否直接使用了另一组浓度轨迹。得到可追溯证据前，不修改 `haloconcentration.py`。

本 Notebook 到此只提交诊断证据，没有自动修改任何主体模块，也没有添加自检模块。